# 课后练习解答（05.04_lora_training）

本解答对应章节课后练习，共 15 题。

### 问题1（单选题）

**题目：** 线性层权重 W0 形状 [4096,4096]，LoRA r=8 时该层新增参数量为？
A. 65,536
B. 4,096
C. 32,768
D. 16,777,216

**解答：** A

**解析：** LoRA 引入 A[r,in] 与 B[out,r]，参数量为 4096×8+8×4096=65,536。


### 问题2（单选题）

**题目：** per_device_train_batch_size=1、gradient_accumulation_steps=8、2 卡并行时等效 batch 为？
A. 16
B. 8
C. 1
D. 64

**解答：** A

**解析：** 等效 batch = 每卡 batch × 累积步数 × 卡数 = 1×8×2=16。


### 问题3（多选题）

**题目：** LoRA 降低训练显存的原因包括？
A. 冻结原始权重，不保存其优化器状态
B. 仅低秩矩阵参与梯度计算
C. 减少需要保存的梯度数量
D. 激活值完全不需要保存

**解答：** ABC

**解析：** LoRA 不减少前向激活，激活仍是显存的重要组成部分。


### 问题4（多选题）

**题目：** 影响 LoRA 表达能力的配置包括？
A. r
B. alpha
C. target_modules
D. lora_dropout

**解答：** ABCD

**解析：** 秩、缩放、作用模块与正则共同决定 LoRA 的表达与泛化。


### 问题5（判断题）

**题目：** LoRA 的缩放系数为 alpha/r，改变 r 时需同步考虑 alpha。

**解答：** 对

**解析：** 实际权重更新为 (alpha/r)*BAx，保持 alpha/r 可控制更新尺度。


### 问题6（判断题）

**题目：** r 越大一定带来更高准确率。

**解答：** 错

**解析：** r 过大会增加参数量与过拟合风险，不一定提升泛化。


### 问题7（填空题）

**题目：** 本实验 q_proj/v_proj 上 LoRA 可训练参数约为 ____，占总参数比例约 ____。

**解答：** 393 万（3,932,160）；0.057%


### 问题8（填空题）

**题目：** SFTTrainer 中 max_seq_length 决定训练样本截断/填充后的 ____。

**解答：** 序列长度上限


### 问题9（简答题）

**题目：** 为什么只选择 q_proj 与 v_proj 而不是全部线性层？

**解答：** 已有研究表明 Q/V 投影对指令跟随和语义理解更关键；只适配两个投影能显著减少可训练参数与优化器状态，降低显存和过拟合风险。


### 问题10（简答题）

**题目：** bf16 与 LoRA 如何协同降低单卡训练 7B 模型的显存压力？

**解答：** bf16 将权重与激活从 fp32 降为 2 字节；LoRA 冻结原始权重，使优化器状态与梯度只存在于低秩适配器上，两者叠加后显存从全参微调的不可行降到单卡可训。


### 问题11（代码设计题）

**题目：** 写出 PEFT LoRA 配置与 model.print_trainable_parameters() 检查片段。

**解答：** ```python
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
```


### 问题12（单选题）

**题目：** 训练 loss 下降但验证 loss 上升，首先应？
A. 降低 r 或增大 dropout，并回滚到最佳 checkpoint
B. 增大 r
C. 提高学习率
D. 移除 LoRA

**解答：** A

**解析：** 过拟合信号下应降低模型容量或增强正则，而不是继续增大容量。


### 问题13（多选题）

**题目：** SwanLab 可用于？
A. 绘制 loss/lr 曲线
B. 查看梯度范数
C. 多组实验对比
D. 替代模型保存

**解答：** ABC

**解析：** SwanLab 是训练可视化平台，不能替代 checkpoint 保存。


### 问题14（判断题）

**题目：** 同时设置 max_steps 与 num_train_epochs 时，max_steps 优先控制训练步数。

**解答：** 对

**解析：** Trainer 中 max_steps 非 None 时直接决定总步数，epochs 不再作为停止条件。


### 问题15（简答题）

**题目：** 对比 r=8 与 r=32 在参数量、显存、过拟合风险和收敛速度上的差异。

**解答：** r=32 的可训练参数和优化器状态约为 r=8 的 4 倍，显存更高；表达力更强但数据不足时更易过拟合；r=8 收敛更快、更省显存，适合小数据微调，r=32 适合数据充足场景。
